<a href="https://colab.research.google.com/github/kinshukpcs24-ux/BIS_Lab/blob/main/1BM24CS137_Cuckoo_Search_ML_Hyperparameter_Optimization/1BM24CS137_Cuckoo_Search_ML_Hyperparameter_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
from scipy.special import gamma
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

# Helper function to calculate fitness using cross-validation
def calculate_fitness(hyperparams, X, y):
    # Ensure parameters are within valid logical ranges
    learning_rate = max(0.0001, min(1.0, hyperparams['learning_rate']))
    batch_size = max(1, int(round(hyperparams['batch_size'])))
    num_trees = max(1, int(round(hyperparams['num_trees'])))

    # Using RandomForestClassifier as the machine learning model
    model = RandomForestClassifier(
        n_estimators=num_trees,
        max_depth=batch_size,
        random_state=42
    )

    # 3-Fold Cross-Validation to evaluate validation accuracy
    scores = cross_val_score(model, X, y, cv=3, scoring='accuracy')
    return np.mean(scores)

# Helper function to generate Levy flight changes for each hyperparameter
def levy_flight_update(nest, best_nest, ranges, beta=1.5):
    # Calculate step size using Mantegna's algorithm for Levy distribution
    sigma_u = (gamma(1 + beta) * np.sin(np.pi * beta / 2) /
               (gamma((1 + beta) / 2) * beta * (2**((beta - 1) / 2))))**(1 / beta)
    sigma_v = 1

    u = np.random.normal(0, sigma_u)
    v = np.random.normal(0, sigma_v)
    step = u / (abs(v)**(1 / beta))

    # Generate a new candidate nest
    new_nest = nest.copy()

    # Apply Levy flight to update each hyperparameter
    for param in ['learning_rate', 'batch_size', 'num_trees']:
        # Step proportional to the distance to the current best solution
        step_size = 0.01 * step * (nest[param] - best_nest[param])
        updated_val = nest[param] + step_size * np.random.normal()

        # Clip parameters within their specific bounding ranges
        new_nest[param] = max(ranges[param][0], min(ranges[param][1], updated_val))

    return new_nest

# Helper function to replace lower fitness nests with probability pa
def replace_worst_nests(nests, pa, ranges):
    num_nests = len(nests)
    # Replaces a fraction (pa) of the worst nests with new random nests
    for i in range(int(num_nests * pa)):
        new_egg = {
            'learning_rate': np.random.uniform(ranges['learning_rate'][0], ranges['learning_rate'][1]),
            'batch_size': np.random.uniform(ranges['batch_size'][0], ranges['batch_size'][1]),
            'num_trees': np.random.uniform(ranges['num_trees'][0], ranges['num_trees'][1]),
            'fitness': 0.0
        }
        nests[i] = new_egg
    return nests

# Main Cuckoo Search Optimizer Function
def cuckoo_search_optimization(N, ranges, max_iter, pa, beta=1.5):
    # Generate synthetic dataset for classification
    X, y = make_classification(n_samples=100, n_features=10, random_state=42)

    # Initialize N nests with random parameters
    nests = []
    for _ in range(N):
        egg = {
            'learning_rate': np.random.uniform(ranges['learning_rate'][0], ranges['learning_rate'][1]),
            'batch_size': np.random.uniform(ranges['batch_size'][0], ranges['batch_size'][1]),
            'num_trees': np.random.uniform(ranges['num_trees'][0], ranges['num_trees'][1]),
            'fitness': 0.0
        }
        egg['fitness'] = calculate_fitness(egg, X, y)
        nests.append(egg)

    # Initial sorting of nests based on fitness
    nests = sorted(nests, key=lambda x: x['fitness'])

    # Main optimization loop
    for iteration in range(max_iter):
        # 1. Levy flight update for each nest
        best_nest = nests[-1]
        for i in range(N):
            new_egg = levy_flight_update(nests[i], best_nest, ranges, beta)
            new_egg['fitness'] = calculate_fitness(new_egg, X, y)

            # If the new nest is better than the current nest, replace it
            if new_egg['fitness'] > nests[i]['fitness']:
                nests[i] = new_egg

        # 2. Sort nests based on fitness
        nests = sorted(nests, key=lambda x: x['fitness'])

        # 3. Replace lower fitness nests
        nests = replace_worst_nests(nests, pa, ranges)

        # Re-evaluate and re-sort after replacements
        for egg in nests:
            egg['fitness'] = calculate_fitness(egg, X, y)
        nests = sorted(nests, key=lambda x: x['fitness'])

    # The optimal solution is at nests[n-1] (nests[-1] in python indices)
    optimal_nest = nests[N - 1]
    return optimal_nest

# Dynamic User Inputs
try:
    print("--- CUCKOO SEARCH HYPERPARAMETER OPTIMIZATION CONFIGURATION ---")
    N_nests = int(input("Enter number of nests (N) [e.g. 10]: "))

    print("\nEnter ranges for hyperparameters (min, max):")
    lr_min = float(input("  learning_rate min [e.g. 0.001]: "))
    lr_max = float(input("  learning_rate max [e.g. 0.5]: "))

    bs_min = float(input("  batch_size (max_depth) min [e.g. 2]: "))
    bs_max = float(input("  batch_size (max_depth) max [e.g. 20]: "))

    nt_min = float(input("  num_trees min [e.g. 10]: "))
    nt_max = float(input("  num_trees max [e.g. 150]: "))

    iterations = int(input("\nEnter number of iterations [e.g. 10]: "))
    prob_abandon = float(input("Enter probability of finding (pa) [e.g. 0.25]: "))
    levy_beta = float(input("Enter Levy parameter (beta) [e.g. 1.5]: "))

    parameter_ranges = {
        'learning_rate': (lr_min, lr_max),
        'batch_size': (bs_min, bs_max),
        'num_trees': (nt_min, nt_max)
    }

    # Run optimizer
    best_solution = cuckoo_search_optimization(
        N=N_nests,
        ranges=parameter_ranges,
        max_iter=iterations,
        pa=prob_abandon,
        beta=levy_beta
    )

    print("\nOptimal Nest Hyperparameters:", best_solution)
except ValueError as e:
    print("\nInvalid input detected. Please run the cell again and enter valid numbers.")

--- CUCKOO SEARCH HYPERPARAMETER OPTIMIZATION CONFIGURATION ---
Enter number of nests (N) [e.g. 10]: 10

Enter ranges for hyperparameters (min, max):
  learning_rate min [e.g. 0.001]: 0.001
  learning_rate max [e.g. 0.5]: 0.5
  batch_size (max_depth) min [e.g. 2]: 2
  batch_size (max_depth) max [e.g. 20]: 20
  num_trees min [e.g. 10]: 10
  num_trees max [e.g. 150]: 150

Enter number of iterations [e.g. 10]: 5
Enter probability of finding (pa) [e.g. 0.25]: 0.2
Enter Levy parameter (beta) [e.g. 1.5]: 1.5

Optimal Nest Hyperparameters: {'learning_rate': 0.2360945887624864, 'batch_size': 10.217987063518857, 'num_trees': 107.09807638029375, 'fitness': np.float64(0.9500891265597149)}
